In [2]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [3]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [4]:
def reload_all():
    import importlib
    import pg_schema.matcher as matcher
    import pg_schema.validator as validator
    import pg_schema.precheck as precheck

    importlib.reload(matcher)
    importlib.reload(validator)
    importlib.reload(precheck)


In [5]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud.json")

In [ ]:
# from pg_schema.loader import SchemaLoader

# schema = SchemaLoader("../../dtgraph/pg_schema/schemas/schema_fraud.json")

##### Rules

In [8]:
Rule1 = Rule('''
MATCH (c:Client)
WHERE NOT c:Mule
WHERE c.name NOT null
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns),
    names = c.name,
    namess = c.name,
    namesss = c.name,
})
''', env=env, type_strict=True)

Rule2 = Rule('''
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
''', env=env, type_strict=True)

Rule3 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHIN]->((t.id):CashIn {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule4 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHOUT]->(tx = (t.id):CashOut {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule5 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)
''', env=env, type_strict=True)

Rule6 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_TRANSFER]->(tx = (t.id):Transfer)
''', env=env, type_strict=True)

Rule7 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_DEBITS]->(tx = (t.id):Debit)
''', env=env, type_strict=True)

In [16]:
Rule_complex1 = Rule('''
MATCH (a:Client)-[:PERFORMED]->(t:Payment)<-[:PERFORMED]-(b:Client)
WITH a, b, t LIMIT 2000
GENERATE
(x = (a.id):Person {
    id = a.id,
    name = a.name,
    name_camel_case = "Text"                 
})-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)<-[():PERFORMED_PAYMENT]-(y = (b.id):Persons {
    id = b.id,
    name = b.name,
    name_camel_case = "Text"
})
''', env=env, type_strict=True)


Rule_complex2 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = "Text"
})-[():PERFORMED_CASHOUT]->(tx = (t.id):CashOut {
    original_amount = t.amount,
    formatted_amount = round(t.amount * 100) / 100.0,
    is_large_amount = t.amount > 100000
})
''', env=env, type_strict=True)

Rule_complex3 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(p:Payment),
      (c)-[:PERFORMED]->(t:Transfer)
WITH c, p, t LIMIT 2000
GENERATE
(x = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = "fake"                 
})-[():PERFORMED_PAYMENT]->(pay = (p.id):Payment),
(x)-[():PERFORMED_TRANSFER]->(tr = (t.id):Transfer)
''', env=env, type_strict=True)

Rule_complex5 = Rule('''
MATCH (c:Client)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
WITH c,
     collect(DISTINCT e.email) AS emails,
     collect(DISTINCT p.phoneNumber) AS phones
LIMIT 2000
GENERATE
(x = (c.id):Person {
    id = c.id,
    name = c.name,
    email = head(emails),
    phone = head(phones),
    name_camel_case = apoc.text.upperCamelCase(c.name)
})
''', env=env, type_strict=True)


Rule_complex6 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(ci:CashIn),
      (c)-[:PERFORMED]->(co:CashOut)
WITH c, ci, co LIMIT 2000
GENERATE
(x = (c.id):Person {
    id = c.id,
    name = c.name
})-[():PERFORMED_CASHIN]->(cin = (ci.id):CashIn {
    original_amount = ci.amount,
    formatted_amount = round(ci.amount * 100) / 100.0,
    is_large_amount = ci.amount > 50000
}),
(x)-[():PERFORMED_CASHOUT]->(cout = (co.id):CashOut {
    original_amount = co.amount,
    formatted_amount = round(co.amount * 100) / 100.0,
    is_large_amount = co.amount > 50000
})
''', env=env, type_strict=True)


Rule_complex_fail1 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):Person { id = c.id, name = c.name })-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)
''', env=env, type_strict=True)

In [17]:
Rule_invalid_target = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(x = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = "fake"                       
})-[():PERFORMED_PAYMENT {
    amount = c.amount
    }]->(tx = (t.id):Payment {
    id = t.id,
    name = "fake",
    name_camel_case = "fake"
})
''', env=env, type_strict=True)

##### Schema Conformance

In [17]:
# Schema checking

from pg_schema.loader import SchemaLoader

schema = SchemaLoader("../../dtgraph/pg_schema/schemas/schema_fraud_copy.json")

from dtgraph.pg_schema.check_schema import check_schema

check_schema([Rule_complex1], schema)


--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (a:Client)-[:PERFORMED]->(t:Payment)<-[:PERFORMED]-(b:Client)\nWITH a, b, t LIMIT 2000', 'constructors': [{'src': {'alias': 'x', 'ids': ['a.id'], 'labels': ['Person'], 'properties': [{'key': 'id', 'value': 'a.id'}, {'key': 'name', 'value': 'a.name'}, {'key': 'name_camel_case', 'value': '"Text"                 \n'}]}, 'edge': {'ids': [], 'labels': ['PERFORMED_PAYMENT']}, 'tgt': {'alias': 'tx', 'ids': ['t.id'], 'labels': ['Payment']}}, {'src': {'alias': 'y', 'ids': ['b.id'], 'labels': ['Persons'], 'properties': [{'key': 'id', 'value': 'b.id'}, {'key': 'name', 'value': 'b.name'}, {'key': 'name_camel_case', 'value': '"Text"\n'}]}, 'edge': {'ids': [], 'labels': ['PERFORMED_PAYMENT']}, 'tgt': {'alias': 'tx'}}]}

All rules conform to provided schema



##### Type Checking

In [ ]:
# Type checking

from dtgraph.type_checking.check_types import check_types

check_types([Rule1, Rule2, Rule3, Rule4, Rule5, Rule6, Rule7], env)

##### Applying Rules

In [ ]:
my_transform = Transformation([Rule1])
my_transform.apply_on(graph)

##### Abort Transformation

In [ ]:
my_transform.abort()